# Ü6.1 — Verschachteltes JSON relationalisieren (LÖSUNG)

`raw.customers` einlesen (geschachtelt: `address`, `contacts[]`), den
mischtypigen `loyalty_points` mit `resolveChoice` vereinheitlichen, dann mit
`Relationalize` in flache Tabellen zerlegen (Root + `contacts`-Child) und beide
als Parquet schreiben.

**Voraussetzung:** Crawler hat `raw.customers` katalogisiert.
`loyalty_points` ist absichtlich mischtypig (`1200` vs `"gold"`) → ohne
`resolveChoice` bekommt man einen `choice`-Typ.

In [ ]:
%idle_timeout 60
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2
# In Glue Studio wird die IAM-Rolle in der UI gewählt. Alternativ per Magic:
# %iam_role arn:aws:iam::<account>:role/AWSGlueServiceRole-GfuGlueTraining
%%configure
{ "--enable-glue-datacatalog": "true" }

In [ ]:
from awsglue.context import GlueContext
from awsglue.transforms import ApplyMapping, Filter
from pyspark.context import SparkContext

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

## 1) Lesen — geschachteltes JSON

In [ ]:
customers = glueContext.create_dynamic_frame.from_catalog(
    database="raw", table_name="customers", transformation_ctx="customers"
)
# Der choice-Typ auf loyalty_points ist hier gut sichtbar:
customers.printSchema()

## 2) resolveChoice — loyalty_points vereinheitlichen

In [ ]:
# loyalty_points vereinheitlichen: cast-nach-string haelt beide Auspraegungen
# (1200 und "gold") verlustfrei. Alternativen: make_struct, project:long.
resolved = customers.resolveChoice(specs=[("loyalty_points", "cast:string")])
resolved.printSchema()

## 3) Relationalize — in flache Frames zerlegen

In [ ]:
# Relationalize zerlegt das geschachtelte JSON in eine Sammlung flacher Frames:
# den Root-Frame plus je einen Child-Frame pro Array (hier contacts).
TEMP = "s3://gfu-glue-training-629452195361/temp/relationalize/"
frames = resolved.relationalize(root_table_name="customers", staging_path=TEMP)
print("erzeugte Frames:", frames.keys())

## 4) Root- und Child-Frame herausgreifen

In [ ]:
root = frames.select("customers")
contacts = frames.select("customers_contacts")
root.printSchema()
contacts.printSchema()

## 5) Beide als Parquet schreiben

In [ ]:
OUT = "s3://gfu-glue-training-629452195361/processed/"
root.toDF().write.mode("overwrite").parquet(OUT + "customers_root/")
contacts.toDF().write.mode("overwrite").parquet(OUT + "customers_contacts/")
print("root rows:", root.count(), "| contacts rows:", contacts.count())